# DeepShield — Member 4 Demo (Colab)

This prototype accepts an uploaded MP4 and runs an **exploratory audio–lip synchronization analysis** using the same basic method as the Member 3 notebook: RMS-based audio activity, MediaPipe lip landmarks, and Pearson correlation.

**Important limitation:** the EfficientNet checkpoint was not provided, so this notebook does not make a visual real/fake prediction and does not claim a validated Trust Score. The visual branch is clearly marked unavailable. Correlation is an experimental signal, not proof that a clip is real or fake.

Run the cells from top to bottom. When the final cell starts, open the temporary Gradio link.

In [ ]:
%pip install -q gradio opencv-python-headless mediapipe==0.10.21 librosa scipy pandas matplotlib
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
import os, subprocess, tempfile, math, warnings
import cv2
import numpy as np
import pandas as pd
import librosa
import mediapipe as mp
import gradio as gr
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from pathlib import Path

print('OpenCV:', cv2.__version__)
print('MediaPipe:', mp.__version__)
print('Gradio:', gr.__version__)
print('Ready. Upload a short MP4 (recommended: 5–20 seconds).')

## Analysis function
This follows the current team method. It samples video frames for speed, so the result is a prototype measure rather than a validated detector.

In [ ]:
def analyze_video(video_path):
    if not video_path or not os.path.exists(video_path):
        return 'Please upload an MP4 video.', None

    workdir = tempfile.mkdtemp(prefix='deepshield_')
    wav_path = os.path.join(workdir, 'audio.wav')
    cmd = ['ffmpeg', '-y', '-i', video_path, '-vn', '-ac', '1', '-ar', '16000', wav_path]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if proc.returncode != 0 or not os.path.exists(wav_path):
        return 'Audio extraction failed. Try a different MP4 with an audio track.', None

    audio, sr = librosa.load(wav_path, sr=16000, mono=True)
    if len(audio) < 2048:
        return 'The audio track is too short to analyze.', None

    rms = librosa.feature.rms(y=audio, frame_length=2048, hop_length=512)[0]
    speech_activity = (rms > np.mean(rms)).astype(float)
    audio_times = librosa.frames_to_time(np.arange(len(speech_activity)), sr=sr, hop_length=512)

    # Extract upper/lower lip landmark distance. Sample up to ~12 frames/second.
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or total_frames <= 0:
        cap.release()
        return 'Could not read the video frame rate or frames.', None

    duration = total_frames / fps
    sample_every = max(1, int(round(fps / 12)))
    frame_times, mouth_signal = [], []
    mp_face_mesh = mp.solutions.face_mesh
    with mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1,
                               refine_landmarks=True, min_detection_confidence=0.5,
                               min_tracking_confidence=0.5) as face_mesh:
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % sample_every == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                result = face_mesh.process(rgb)
                if result.multi_face_landmarks:
                    lm = result.multi_face_landmarks[0].landmark
                    # Same lip landmark pair as the team's notebook: 13 and 14.
                    mouth = abs(lm[14].y - lm[13].y)
                else:
                    mouth = np.nan
                frame_times.append(frame_idx / fps)
                mouth_signal.append(mouth)
            frame_idx += 1
    cap.release()

    frame_times = np.asarray(frame_times, dtype=float)
    mouth_signal = np.asarray(mouth_signal, dtype=float)
    valid = np.isfinite(mouth_signal)
    face_detection_rate = float(valid.mean()) if len(valid) else 0.0
    if valid.sum() < 5:
        return (f'Analysis incomplete: a face/lip signal was detected in too few sampled frames.\n'
                f'Duration: {duration:.1f}s | Face/lip detection rate: {face_detection_rate:.0%}\n'
                'Visual authenticity: NOT AVAILABLE (model checkpoint not supplied).', None)

    # Fill missing landmark values by interpolation, then align mouth signal to audio times.
    mouth_signal = np.interp(frame_times, frame_times[valid], mouth_signal[valid])
    mouth_interp = np.interp(audio_times, frame_times, mouth_signal)
    mouth_norm = (mouth_interp - mouth_interp.min()) / (mouth_interp.max() - mouth_interp.min() + 1e-8)

    if np.std(speech_activity) < 1e-8 or np.std(mouth_norm) < 1e-8:
        corr, p_value = 0.0, 1.0
        corr_note = 'Signal variation was too low for a meaningful correlation.'
    else:
        corr, p_value = pearsonr(speech_activity.astype(float), mouth_norm.astype(float))
        corr = float(np.nan_to_num(corr))
        p_value = float(np.nan_to_num(p_value, nan=1.0))
        corr_note = 'Correlation describes co-variation; it is not a calibrated authenticity probability.'

    preliminary_score = max(0.0, corr) * 100.0
    # Evidence-only result; deliberately no real/fake verdict or trust percentage.
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(audio_times, speech_activity, label='Audio activity (RMS threshold)', linewidth=1.2)
    ax.plot(audio_times, mouth_norm, label='Normalized lip movement', linewidth=1.2)
    ax.set_xlabel('Time (seconds)'); ax.set_ylabel('Normalized signal')
    ax.set_title('Exploratory Audio–Lip Signal Comparison'); ax.grid(True, alpha=0.25); ax.legend()
    fig.tight_layout()

    report = (
        'DEEPSHIELD — PRELIMINARY ANALYSIS\n'
        f'Video duration: {duration:.1f} seconds\n'
        f'Sampled frames: {len(frame_times)} (approx. 12 frames/sec maximum)\n'
        f'Face/lip detection rate: {face_detection_rate:.0%}\n'
        f'Audio–lip Pearson correlation: {corr:.4f}\n'
        f'P-value (exploratory): {p_value:.4g}\n'
        f'Preliminary synchronization index: {preliminary_score:.1f}/100\n\n'
        'Visual authenticity: NOT AVAILABLE — EfficientNet checkpoint was not supplied.\n'
        'Final Trust Score: NOT CALCULATED — multimodal fusion is not implemented/validated.\n\n'
        'Interpretation: Do not classify this clip as real or fake from this score alone.\n'
        + corr_note
    )
    return report, fig


## Launch the demo
Run the cell, wait for the Gradio public link, then open it. The link is temporary and only available while the Colab runtime is active.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title='DeepShield Prototype') as demo:
    gr.Markdown('# 🛡️ DeepShield\n### Multimodal Deepfake Analysis — Research Prototype')
    gr.Markdown('Upload a short video to inspect audio–lip synchronization cues. The visual classifier is unavailable because its trained checkpoint was not supplied.')
    with gr.Row():
        video = gr.Video(label='Upload video (MP4)', sources=['upload'], format='mp4')
        with gr.Column():
            run_btn = gr.Button('Analyze video', variant='primary')
            result = gr.Textbox(label='Analysis report', lines=14)
    plot = gr.Plot(label='Audio–lip signal plot')
    run_btn.click(fn=analyze_video, inputs=video, outputs=[result, plot])
    gr.Markdown('**Limitations:** this is not a validated deepfake detector. It uses a simple RMS speech-activity signal and lip-opening signal; background music, silence, multiple speakers, off-screen speech, occlusion, and face-tracking errors can affect the correlation.')

demo.launch(share=True, debug=False)